# Project 1: Dynamic Pricing Predictor for Ride-Sharing Apps
## Milestone 2: Regression Modeling, Evaluation & Price Driver Interpretation
**Author:** Kamaljit  
**Track:** AI & ML Minor Project  
**Platform:** Nyeras LMS  

---

### Objectives
1. Train and benchmark five competitive regression algorithms:
   - **Linear Regression (Ordinary Least Squares)**
   - **Ridge Regression (L2 Regularization)**
   - **Lasso Regression (L1 Feature Sparsity)**
   - **Random Forest Regressor (Non-linear Bagging Ensemble)**
   - **Gradient Boosting Regressor (Sequential Gradient Boosting)**
2. Evaluate model performance using standard industry metrics: **RMSE ($)**, **MAE ($)**, and **$R^2$ Score**.
3. Identify and explain the **biggest price drivers** (surge factor, trip distance, adverse weather, rush hour).
4. Export the winning model for interactive deployment.

In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

print('Libraries loaded successfully.')

### 1. Ingesting Engineered Features

In [2]:
feat_path = os.path.join('..', 'data', 'processed', 'features_engineered.csv')
df = pd.read_csv(feat_path)
print(f'Engineered Dataset: {df.shape[0]} samples, {df.shape[1]} columns')

# Encode categorical columns
df_encoded = pd.get_dummies(df, columns=['weather_condition', 'time_bucket', 'pickup_location'], drop_first=True)

# Target and Feature separation
drop_cols = ['ride_id', 'timestamp', 'datetime', 'dropoff_location', 'day_of_week', 'price', 'base_fare']
feature_cols = [c for c in df_encoded.columns if c not in drop_cols]

X = df_encoded[feature_cols]
y = df_encoded['price']
print(f'Matrix X: {X.shape}, Target y: {y.shape}')

### 2. Train-Test Split (80% Train, 20% Held-Out Test)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f'Training Set: {X_train.shape[0]} samples | Test Set: {X_test.shape[0]} samples')

### 3. Model Training & Comparison

In [4]:
models = {
    'Linear Regression': Pipeline([('scaler', StandardScaler()), ('reg', LinearRegression())]),
    'Ridge Regression': Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=1.0))]),
    'Lasso Regression': Pipeline([('scaler', StandardScaler()), ('reg', Lasso(alpha=0.05, max_iter=2000))]),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=120, learning_rate=0.08, max_depth=5, random_state=42)
}

results = []
preds = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    preds[name] = y_pred
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({'Model': name, 'MAE ($)': round(mae, 2), 'RMSE ($)': round(rmse, 2), 'R2 Score': round(r2, 4)})

benchmark_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False).reset_index(drop=True)
benchmark_df

### 4. Evaluating Errors (RMSE & MAE Comparison Plot)

In [5]:
plt.figure(figsize=(10, 5))
bar_w = 0.35
idx = np.arange(len(benchmark_df))
plt.bar(idx - bar_w/2, benchmark_df['RMSE ($)'], width=bar_w, label='RMSE ($)', color='#e74c3c')
plt.bar(idx + bar_w/2, benchmark_df['MAE ($)'], width=bar_w, label='MAE ($)', color='#3498db')
plt.xticks(idx, benchmark_df['Model'], rotation=15, ha='right')
plt.ylabel('Dollars ($)')
plt.title('Benchmark Error Comparison Across Models', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()

### 5. Explaining the Biggest Price Drivers (Feature Importance)
Which factors cause dynamic ride prices to fluctuate most significantly?

In [6]:
rf_model = models['Random Forest']
feat_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 6))
top10 = feat_imp.head(10).iloc[::-1]
plt.barh(top10['Feature'], top10['Importance'], color='#2ecc71', edgecolor='#27ae60')
plt.xlabel('Importance Weight')
plt.title('Top 10 Biggest Price Drivers in Dynamic Ride Pricing', fontsize=13, fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 6. Actual vs Predicted Residual Distribution

In [7]:
best_model_name = benchmark_df.iloc[0]['Model']
best_y_pred = preds[best_model_name]

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_y_pred, alpha=0.35, color='#2980b9')
min_p, max_p = min(y_test.min(), best_y_pred.min()), max(y_test.max(), best_y_pred.max())
plt.plot([min_p, max_p], [min_p, max_p], 'r--', lw=2, label='Perfect Fit (y=x)')
plt.xlabel('Actual Fare ($)')
plt.ylabel('Predicted Fare ($)')
plt.title(f'Actual vs Predicted Ride Prices ({best_model_name})', fontsize=13, fontweight='bold')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()